# great.py

### from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments

#### AutoTokenizer 

(https://www.linkedin.com/pulse/understanding-autotokenizer-huggingface-transformers-m-shivanandhan-zfbsc/)
Tokenizers break down text into smaller units called tokens.
AutoTokenizer is a special class in the Huggingface Transformers library. It helps choose the right tokenizer for the model without knowing the details by calling .from_pretrained("gpt2").

The tokenizer breaks the text into tokens and converts them into numbers. You'll see an output like below

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "I love Armenia"
tokens = tokenizer(text)

In [7]:
# input_ids are the token IDs.
# attention_mask tells the model which tokens to pay attention to (1 means pay attention, 0 means ignore).

print(tokens)

{'input_ids': [40, 1842, 32369], 'attention_mask': [1, 1, 1]}


In [5]:
# To decode the token IDs to text
decoded_text = tokenizer.decode(tokens['input_ids'])
print(decoded_text)

I love Armenia


In [4]:
# To see which tokenizer was used
print(type(tokenizer))

<class 'transformers.models.gpt2.tokenization_gpt2_fast.GPT2TokenizerFast'>


#### AutoModelForCausalLM

AutoModelForCausalLM class is designed to automatically select and load a model architecture that is suitable for casual language modeling, based on the model name or path you provide (for example 'gpt2'). It is useful when you want to load a pretrained model without needing to know the specific model architecture in advance.

Causal Language Modeling is a task where the model generates text by predicting the next token in a sequence, given the previous tokens.

The *from_pretrained* class method is used to load a pretrained model from the Hugging Face Model Hub or from a local directory. This model automatically determines the correct model architecture based on the input and loads the model with pretrained weights.
1. You provide the model identifier (e.g. 'gpt2') and the method identifies the correct model architecture (e.g. 'GPT2LMHeadModel) based on the identifier
2. The model downloads the model's configuration, vocabulary and pretrained weights from the Hugging Face Model Hub and loads the model.

*Note:* 
- Model Identifier is a string (typically the name or the path) that uniquely identifies a specific pretrained model, it gives the transformers library a hint which model you want to load. E.g. 'gpt2' refers to a specific pretrained model of GPT-2 available on the Hugging Face Model Hub.
- Model Architecture refers to the specific neural network structure or design used by the model. It defines how the layers in the model are arranged, howthey interact and how the data flows through the model. E.g. 'GPT2LMHeadModel is the correct architecture for the 'gpt2' identifier

In [8]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("gpt2")
print(type(model))

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>


#### TrainingArguments

TrainingArguments is a class that holds all the hyperparameters and configuration oprtions needed to train a model. It is a central component when using the Trainer API, which simplifies the process of training and evaluating models.
It encapsulates various hyperparameters such as learning rate, batch size, number of epochs, etc that are essential for training, also configuartion like gradient accumulation, weight decay, the directory where the logs are stored, how often to save the model checkpoint and also helps in managing the computational resources by specifying how many devides (GPU/TPU/CPU) to use, and whether to use mixed precision (FP16) training.

And these arguments are later used in the Trainer.

### GReaT

The GReaT class handles the whole generation flow. It is used to fine-tune a large language model for tabular data, and to sample synthetic tabular data.

#### LoRA fine-tuning

*Nice explanation of the fine-tuning and LoRA*: [https://medium.com/@kailash.thiyagarajan/fine-tuning-large-language-models-with-lora-demystifying-efficient-adaptation-25fa0a389075]


There are various methods of fine tuning

**Layer-wise Fine Tuning:** This is the most commonly used method where either the early layers of the pre-trained model (closest to input) are frozen, and only the later layers are fine-tuned or the entire model is fine tuned.

**Parameter Selective Fine-tuning:** This method identifies and updates only a subset of parameters deemed relevant for the task.

**Adapter-based Fine-Tuning:** This technique introduces lightweight “adapter” modules alongside the LLM, containing task-specific adjustments.

**LoRA(Low-Rank Adaptation) Config**
is imported from peft library. Is a method that fine-tunes only a small portion of the model's parameters, making it computationally efficient while still maintaining good performance.

It uses the idea of the *intrinsic dimensionality.* We empirically show that common pre-trained models have a very low intrinsic dimension; in other words, there exists a low dimension reparameterization that is as effective for fine-tuning as the full parameter space. We hypothesize that the change in weights during model adaptaion also has a low 'intrinsic rank', leading to our propsed LoRA.

Intrinsic dimensionality refers to the minimum number of dimensions required to accurately describe the structure or geometry of data within a higher-dimensional space. In other words, it captures the underlying complexity of data, despite the data being embedded in a space of potentially much higher dimemsions.

Here's an anology: Imagine a flat piece of paper (2D) twisted into a complex shape in 3D space. While the paper exists in 3D space, its intrinsic dimensionality is 2D, because all the points on the paper can be described using just two coordinates (x and y).

In [ ]:
# the fine tuning is automatically called
lora_config = LoraConfig(
                r=16,  # only training 0.16% of the parameters of the model
                lora_alpha=32,
                target_modules=[
                    "c_attn"
                ],  # this is specific for gpt2 model, to be adapted, means it is used on attention block
                lora_dropout=0.05, # to prevent overfitting
                bias="none", # no biases (the constants) as they represent s small fraction of the model parameters
                task_type=TaskType.CAUSAL_LM,  # this is specific for gpt2 model, to be adapted

#### sample, great_sample, impute

Purpose is generating synthetic samples of tabular data by using a pre-trained model. Generates n_sample synthetic rows, optionally starting from a specified column

In [ ]:
def sample(
    self,
    n_samples: int,
    start_col: tp.Optional[str] = "",
    start_col_dist: tp.Optional[tp.Union[dict, list]] = None,
    temperature: float = 0.7, # controls the softmax function for token sampling. Lower values make it sharper (0 equals greedy search), higher values introduce more diversity but also uncertainty
    k: int = 100, # Sampling btach size. Higher values speed up the generation process.
    max_length: int = 100,
    drop_nan: bool = False,
    device: str = "cuda",
) -> pd.DataFrame:

**great_sample**
Generating synthetic tabular data samples conditioned on a given input

**impute**
Uses great_sample to impute a DataFrame with missing values usingthe GReaT model

# great.dataset.py

### from datasets import Dataset

Dataset class code is in the [link](https://github.com/huggingface/datasets/blob/2.21.0/src/datasets/arrow_dataset.py#L668)

It is a powerful structure used to handle and manipulate large datasets for ML tasks. It is designed to streamline the process of loading, processing, and using datasets, especially in combination with models from the *transformers* library.

### from transformers import DataCollatorWithPadding

The sourse of the code can be found [here](https://github.com/huggingface/transformers/blob/v4.44.2/src/transformers/data/data_collator.py#L236)

DataCollatorWithPadding is a class in Hugging Face Transformers that helps in preparing batches of data for training transformer models. Specifically, it is designed to handle cases where input sequences have different lengths by dynamically padding them within a batch.

When training a transformer model, especially for BERT or GPT, it isrequired or advisable to have the same length for the batch inputs in order to processthem in parallel efficiently. However, since sequences might have different lengths, they need to be padded to a common length within each batch. The DataCollatorWithPadding class automates this process.

In [3]:
# Initialization: You create an instance of DataCollatorWithPadding, typically specifying the tokenizer 
# to be used and any other relevant parameters.

from transformers import DataCollatorWithPadding, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
# Batch Preparation: When you’re preparing batches of data for training, you pass your list of examples 
# to the data_collator instance.

examples = [
{'input_ids': [1, 2, 3], 'labels': [0]},
{'input_ids': [4, 5], 'labels': [1]},
# …
]

batch = data_collator(examples)
print(batch)

{'input_ids': tensor([[1, 2, 3],
        [4, 5, 0]]), 'labels': tensor([[0],
        [1]]), 'attention_mask': tensor([[1, 1, 1],
        [1, 1, 0]])}


The data_collator will pad the sequences in the batch dynamically to the maximum length within that batch.

Example output of the batch after dynamic padding
{
“input_ids”: [[1, 2, 3, 0], [4, 5, 0, 0]],
“labels”: [[0, -100, -100, -100], [1, -100, -100, -100]],
}

In the output, the 0 values represent the padding tokens, and -100 is a common value used to mask out certain elements during training.

This dynamic padding within each batch helps optimize memory usage and allows for more efficient training of transformer models with variable-length input sequences.

- Instead of padding all sequences in the dataset to a fixed length (which could waste memory or computational resources), DataCollatorWithPadding pads the sequences dynamically to the length of the longest sequence in the current batch for all columns (not for each column its max length). This means that each batch can have different sequence lengths, reducing the amount of unnecessary padding.
- It works in conjuction with the tokenizer, which typically provides the padding token and attention mask that indicate which parts of the input are padding. 

In [1]:
class DataCollatorWithPadding:
    """
    Data collator that will dynamically pad the inputs received.

    Args:
        tokenizer ([`PreTrainedTokenizer`] or [`PreTrainedTokenizerFast`]):
            The tokenizer used for encoding the data.
        padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:

            - `True` or `'longest'` (default): Pad to the longest sequence in the batch (or no padding if only a single
              sequence is provided).
            - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
              acceptable input length for the model if that argument is not provided.
            - `False` or `'do_not_pad'`: No padding (i.e., can output a batch with sequences of different lengths).
        max_length (`int`, *optional*):
            Maximum length of the returned list and optionally padding length (see above).
        pad_to_multiple_of (`int`, *optional*):
            If set will pad the sequence to a multiple of the provided value.

            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
        return_tensors (`str`, *optional*, defaults to `"pt"`):
            The type of Tensor to return. Allowable values are "np", "pt" and "tf".
    """

    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True # you can specify the padding strategy such as 'longest', 'max_length', 'do_not_pad'
    max_length: Optional[int] = None # pad al sequences to a specific max length
    pad_to_multiple_of: Optional[int] = None # if you want the length to be padded to a multiple of some number
    return_tensors: str = "pt"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = pad_without_fast_tokenizer_warning(
            self.tokenizer,
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )
        if "label" in batch:
            batch["labels"] = batch["label"]
            del batch["label"]
        if "label_ids" in batch:
            batch["labels"] = batch["label_ids"]
            del batch["label_ids"]
        return batch

NameError: name 'PreTrainedTokenizerBase' is not defined

### GReaTDataset

GReaTDataset overwrites the _getitem function of the HuggingFace Dataset Class to include the permutation step.

Specifically these are the functions that were changed
- set_tokenizer was added to have the tokenizer from transformers.Autotokenizer
- getitem which on the original is to retrive an instance or slice of rows from the dataset, now it includes also a step for permuting the columns by shuffling them
- getitems


In [9]:
class GReaTDataset(Dataset):
    """GReaT Dataset

    The GReaTDataset overwrites the _getitem function of the HuggingFace Dataset Class to include the permutation step.

    Attributes:
        tokenizer (AutoTokenizer): Tokenizer from HuggingFace
    """

    # Assigns a tokenizer to the dataset. The tokenizer is exepcted to be an instance of Hugging Face's 'transformers.AutoTokenizer'
    def set_tokenizer(self, tokenizer):
        """Set the Tokenizer

        Args:
            tokenizer: Tokenizer from HuggingFace
        """
        self.tokenizer = tokenizer

    # retrieves an item from the dataset, 
    # shuffles the columns and creates a textual representation    
    # tokenizes the shuffles text using the tokenizer mentioned above
    def _getitem(
        self, key: tp.Union[int, slice, str], decoded: bool = True, **kwargs
    ) -> tp.Union[tp.Dict, tp.List]:
        """Get Item from Tabular Data

        Get one instance of the tabular data, permuted, converted to text and tokenized.
        """
        # If int, what else?
        row = self._data.fast_slice(key, 1)

        shuffle_idx = list(range(row.num_columns))
        random.shuffle(shuffle_idx)

        shuffled_text = ", ".join(
            [
                "%s is %s"
                % (row.column_names[i], str(row.columns[i].to_pylist()[0]).strip())
                for i in shuffle_idx
            ]
        )
        tokenized_text = self.tokenizer(shuffled_text, padding=True)
        return tokenized_text

    # allows indexing with a list of indices, returning a list of tokenized items
    def __getitems__(self, keys: tp.Union[int, slice, str, list]):
        if isinstance(keys, list):
            return [self._getitem(key) for key in keys]
        else:
            return self._getitem(keys)

NameError: name 'Dataset' is not defined

### GReatDataCollator

Overwrites the DataCollatorWithPadding to also pad the labels and not only the input_ids

In [ ]:
class GReaTDataCollator(DataCollatorWithPadding):
    """GReaT Data Collator

    Overwrites the DataCollatorWithPadding to also pad the labels and not only the input_ids
    """
    # pads the input features and also ensures that the labels are padded to the same length as input_ids
    def __call__(self, features: tp.List[tp.Dict[str, tp.Any]]):
        batch = self.tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )
        batch["labels"] = batch["input_ids"].clone()
        return batch


In [ ]:
# This is the original function
def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = pad_without_fast_tokenizer_warning(
            self.tokenizer,
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )
        if "label" in batch:
            batch["labels"] = batch["label"]
            del batch["label"]
        if "label_ids" in batch:
            batch["labels"] = batch["label_ids"]
            del batch["label_ids"]
        return batch

The difference
1. In the original the code calls *pad_without_fast_tokenizer_warning*, which is likely a custom wrapper to handle padding without triggering a warning related to the fast tokenizer. And also it renames different label keys ('label' or 'label_ids') to a unified 'labels' ensuring consisting output
2. In the modified we use the Hugging Face *tokenizer.pad* method to perform padding. After padding, the 'input_ids' (the tokenized sequence) are cloned, and the clone is assigned to the 'labels' key in the batch. Why? Because usually 'labels' represent the target output that the model is expected to predict. In many tasks such as text classification, these 'labels' are predefined in the dataset. However for casual lnaguage modeling 'labels' are derived fom the input_ids (the tokenized numbers of the input text). Hence the 'labels' are the same as the input_ids but shifted by one position

The tokenizer.pad is chosen according to the language model we are using. 
So when we call 

In [27]:
# Usually the padding is already selected as a result of using the pretrained model, however we can also customize it 

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

custom_padding_token_id = 1 # pad with 1
tokenizer.pad_token_id = 1

texts = ["Hello", "Hi there!"]
encoded_inputs = tokenizer(texts)
encoded_inputs_p = tokenizer(texts, padding=True)
print('Without Padding: ', encoded_inputs['input_ids'])
print('With Padding: ', encoded_inputs_p['input_ids'])

Without Padding:  [[15496], [17250, 612, 0]]
With Padding:  [[15496, 1, 1], [17250, 612, 0]]


# great_start.py

The classes here are subclasses of GReaTStart. This is an abstract class that sets the foundation forcreating start tokens, but it doesn't implement how the tokens are generated. So it can't be used directly because the *get_start_tokens* method raises an error.

### CategoricalStart

CategoricalStart randomly selects values (categories) from the provided distribution and constructs phrases like 'Color is Red'.

Instead of taking actual rows from a dataset, it uses a distribution of categories ('Red': 0.6, 'Green': 0.3, 'Blue': 0.1) and randomly selects values based on those probabilities.

And it returns n_sample times lists of numbers with same length dues to padding.

*Note*: The padding is applied to make all sequences the same length within batch, so different sentences might have different lengths after tokenization. 

In [1]:
class CategoricalStart(GReaTStart):
    """Categorical Starting Feature

    A categorical column with its categories is used as starting point.

    Attributes:
        start_col (str): Name of the categorical column (e.g 'color')
        population (list[str]): Possible values the column can take (e.g. 'Red', 'Green', 'Blue')
        weights (list[float]): Probabilities for the individual categories (e.g. 'Red': 0.6, 'Green': 0.3, 'Blue': 0.1)

    """

    def __init__(self, tokenizer, start_col: str, start_col_dist: dict):
        """Initializes the Categorical Start

        Args:
            tokenizer: Tokenizer from the HuggingFace library
            start_col: Name of the categorical column (e.g. 'Color')
            start_col_dist: Distribution of the categorical column (dict of form {"Cat A": 0.8, "Cat B": 0.2})
        """
        super().__init__(tokenizer)

        assert isinstance(start_col, str), ""
        assert isinstance(start_col_dist, dict), ""

        self.start_col = start_col # The name of the categorical column
        self.population = list(start_col_dist.keys()) # The possible values that the column can take
        self.weights = list(start_col_dist.values())

    def get_start_tokens(self, n_samples): # n_samples is how many token we want ot generate
        start_words = random.choices(self.population, self.weights, k=n_samples) # randomly picks n_samples from population list
        start_text = [self.start_col + " is " + str(s) + "," for s in start_words]
        start_tokens = _pad_tokens(self.tokenizer(start_text)["input_ids"]) # converted to numbers
        return start_tokens

NameError: name 'GReaTStart' is not defined

### ContinuousStart

Selects random values from the continuous columns and adds noise to them.

E.g. we have a columns with 'Temperature' column where values are [20.5, 21, 22.3, 19.8].The *get_start_tokens* method randomly picks one of these values e.g. 21, adds noise and creates a prompt like 'Temperature is 21.04'. And then converts it to a token.

E.g. we might have [[10, 11, 12, 13], [14, 15, 16, 17]] meaning [[Temperature is 21.04][Temperature is 20.55]]

In [3]:
class ContinuousStart(GReaTStart):
    """Continuous Starting Feature

    A continuous column with some noise is used as starting point.

    Attributes:
        start_col (str): Name of the continuous column
        start_col_dist (list[float]): The continuous column from the train data set
        noise (float): Size of noise that is added to each value
        decimal_places (int): Number of decimal places the continuous values have
    """

    def __init__(
        self,
        tokenizer,
        start_col: str,
        start_col_dist: tp.List[float],
        noise: float = 0.01,
        decimal_places: int = 5,
    ):
        """Initializes the Continuous Start

        Args:
            tokenizer: Tokenizer from the HuggingFace library
            start_col: Name of the continuous column
            start_col_dist: The continuous column from the train data set
            noise: Size of noise that is added to each value
            decimal_places: Number of decimal places the continuous values have
        """
        super().__init__(tokenizer)

        assert isinstance(start_col, str), ""
        assert isinstance(start_col_dist, list), ""

        self.start_col = start_col
        self.start_col_dist = start_col_dist
        self.noise = noise
        self.decimal_places = decimal_places

    def get_start_tokens(self, n_samples):
        start_words = random.choices(self.start_col_dist, k=n_samples)
        # start_words += np.random.normal(size=n_samples) * self.noise  # add noise to start words
        start_text = [
            self.start_col + " is " + format(s, f".{self.decimal_places}f") + ","
            for s in start_words
        ]
        start_tokens = _pad_tokens(self.tokenizer(start_text)["input_ids"])
        return start_tokens

NameError: name 'GReaTStart' is not defined

### RandomStart

Randomly chooses the columns (if there is no distribution for that) and tokenizes it. So if we have columns 'Color', 'Age', 'Name'. It rabdomly chooses and tokenizes so we have [[12, 13, 14], [14, 15, 16]] which can mean 'Age is' 'Name is'

In [ ]:
class RandomStart(GReaTStart):
    """Random Starting Features

    Random column names are used as start point. Can be used if no distribution of any column is known.

    Attributes:
        all_columns (List[str]): Names of all columns
    """

    def __init__(self, tokenizer, all_columns: tp.List[str]):
        """Initializes the Random Start

        Args:
            tokenizer: Tokenizer from the HuggingFace library
            all_columns: Names of all columns
        """
        super().__init__(tokenizer)
        self.all_columns = all_columns

    def get_start_tokens(self, n_samples):
        start_words = random.choices(self.all_columns, k=n_samples)
        start_text = [s + " is " for s in start_words]
        start_tokens = _pad_tokens(self.tokenizer(start_text)["input_ids"])
        return start_tokens

# great_trainer.py

### from transformers import Trainer

The Trainer class in HuggingFace is a high-level API designed to simplify the training and evalutaion of models. It is for feature-complete training in PyTorch. It abstracts away many of the complexities involved in training deep learning models, such as managing data loading, gradient accumulation and model saving.

It goes hand-in-hand with the TrainingArguments class, which offers a wide range of options to customize how a model is trained. Together these 2 classes provide a complete training API.


### _seed_worker

To ensure the randomness is consistent and reproducible

Why is it important?
In multi-worker setups, each worker is responsible for loading a part of the dataset. If you have random operations like shuffling, data augmentation, or random cropping in the dataset loading process, you want these operations to be consistent and reproducible for debugging or experimentation purposes. Without seeding, different workers could produce different outputs each time, leading to non-deterministic behavior.

In [ ]:
def _seed_worker(_):
    """
    Helper function to set worker seed during Dataloader initialization.
    """
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    torch.cuda.manual_seed_all(worker_seed)

### GReaTTrainer

In [ ]:
# Modified version of Trainer class

class GReaTTrainer(Trainer):
    """GReaT Trainer

    Overwrites the get_train_dataloader methode of the HuggingFace Trainer to not remove the "unused" columns -
    they are needed later!
    """

    def get_train_dataloader(self) -> DataLoader:
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")

        data_collator = self.data_collator
        train_dataset = (
            self.train_dataset
        )  # self._remove_unused_columns(self.train_dataset, description="training")
        train_sampler = self._get_train_sampler()

        return DataLoader(
            train_dataset,
            batch_size=self._train_batch_size,
            sampler=train_sampler,
            collate_fn=data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            worker_init_fn=_seed_worker,
        )


In [ ]:
# The original Trainer.get_train_dataloader

def get_train_dataloader(self) -> DataLoader:
        """
        Returns the training [`~torch.utils.data.DataLoader`].

        Will use no sampler if `train_dataset` does not implement `__len__`, a random sampler (adapted to distributed
        training if necessary) otherwise.

        Subclass and override this method if you want to inject some custom behavior.
        """
        if self.train_dataset is None: # checks if a training dataset has been provided
            raise ValueError("Trainer: training requires a train_dataset.")

        train_dataset = self.train_dataset
        data_collator = self.data_collator
        if is_datasets_available() and isinstance(train_dataset, datasets.Dataset): # checks whether it's a Hugging Face dataset
            train_dataset = self._remove_unused_columns(train_dataset, description="training")
        else:
            data_collator = self._get_collator_with_removed_columns(data_collator, description="training") # if it is not a Hugging Face dataset then applied this method

        dataloader_params = {
            "batch_size": self._train_batch_size,
            "collate_fn": data_collator, # to handle batches
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
        }

        if not isinstance(train_dataset, torch.utils.data.IterableDataset):
            dataloader_params["sampler"] = self._get_train_sampler()
            dataloader_params["drop_last"] = self.args.dataloader_drop_last
            dataloader_params["worker_init_fn"] = seed_worker
            dataloader_params["prefetch_factor"] = self.args.dataloader_prefetch_factor

        return self.accelerator.prepare(DataLoader(train_dataset, **dataloader_params))

In [ ]:
# _remove_unused_columns function which was removed from GReaT model.
# The function removes columns from the dataset that are not required by the model's forward method.
# E.g. forward(age, gender, name) here these columns are signature (importants, and the other columns like 'education'.. are ignored)
# This ensures that only relevant data is passed to the model, making the training process more efficient

def _remove_unused_columns(self, dataset: "datasets.Dataset", description: Optional[str] = None):
        if not self.args.remove_unused_columns:
            return dataset
        self._set_signature_columns_if_needed()
        signature_columns = self._signature_columns

        ignored_columns = list(set(dataset.column_names) - set(signature_columns))
        if len(ignored_columns) > 0:
            dset_description = "" if description is None else f"in the {description} set"
            logger.info(
                f"The following columns {dset_description} don't have a corresponding argument in "
                f"`{self.model.__class__.__name__}.forward` and have been ignored: {', '.join(ignored_columns)}."
                f" If {', '.join(ignored_columns)} are not expected by `{self.model.__class__.__name__}.forward`, "
                " you can safely ignore this message."
            )

        columns = [k for k in signature_columns if k in dataset.column_names]
        if len(columns) == 0:
            raise ValueError(
                "No columns in the dataset match the model's forward method signature. "
                f"The following columns have been ignored: [{', '.join(ignored_columns)}]. "
                "Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`."
            )

        if version.parse(datasets.__version__) < version.parse("1.4.0"):
            dataset.set_format(
                type=dataset.format["type"], columns=columns, format_kwargs=dataset.format["format_kwargs"]
            )
            return dataset
        else:
            return dataset.remove_columns(ignored_columns)


# great_utils.py

In [2]:
import typing as tp

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer

### _array_to_dataframe

In [3]:
def _array_to_dataframe(
    data: tp.Union[pd.DataFrame, np.ndarray], columns=None
) -> pd.DataFrame:
    """Converts a Numpy Array to a Pandas DataFrame

    Args:
        data: Pandas DataFrame or Numpy NDArray
        columns: If data is a Numpy Array, columns needs to be a list of all column names

    Returns:
        Pandas DataFrame with the given data
    """
    if isinstance(data, pd.DataFrame):
        return data

    assert isinstance(
        data, np.ndarray
    ), "Input needs to be a Pandas DataFrame or a Numpy NDArray"
    assert (
        columns
    ), "To convert the data into a Pandas DataFrame, a list of column names has to be given!"
    assert len(columns) == len(
        data[0]
    ), "%d column names are given, but array has %d columns!" % (
        len(columns),
        len(data[0]),
    )

    return pd.DataFrame(data=data, columns=columns)

In [18]:
data = np.array([['John', 30, 'New York', 'Engineer'],
                 ['Jane', np.nan, 'San Francisco', 'Artist'],
                 [np.nan, 28, 'New York', 'Doctor'],
                 ['Mike', 40, 'New York', 'Artist']])
columns = ['Name', 'Age', 'City', 'Occupation']
df = _array_to_dataframe(data, columns)
df['Age'] = pd.to_numeric(df["Age"], errors='coerce')
df

,Name,Age,City,Occupation
0,John,30.0,New York,Engineer
1,Jane,NaN,San Francisco,Artist
2,nan,28.0,New York,Doctor
3,Mike,40.0,New York,Artist


### _get_column_distribution

In [19]:
def _get_column_distribution(df: pd.DataFrame, col: str) -> tp.Union[list, dict]:
    """Returns the distribution of a given column. If continuous, returns a list of all values.
        If categorical, returns a dictionary in form {"A": 0.6, "B": 0.4}

    Args:
        df: pandas DataFrame
        col: name of the column

    Returns:
        Distribution of the column
    """
    if df[col].dtype == "float":
        col_dist = df[col].to_list()
    else:
        col_dist = df[col].value_counts(1).to_dict()
    return col_dist

In [20]:
print(_get_column_distribution(df, 'City'))
print(_get_column_distribution(df, 'Age'))

{'New York': 0.75, 'San Francisco': 0.25}
[30.0, nan, 28.0, 40.0]


### _encode_row_partial

In [24]:
# leaves out the ones with non values
def _encode_row_partial(row, shuffle=True):
    """Function that takes a row and converts all columns into the text representation that are not NaN."""
    num_cols = len(row.index)
    if not shuffle:
        idx_list = np.arange(num_cols)
    else:
        idx_list = np.random.permutation(num_cols)

    lists = ", ".join(
        sum(
            [
                [f"{row.index[i]} is {row[row.index[i]]}"]
                if not pd.isna(row[row.index[i]])
                else []
                for i in idx_list
            ],
            [],
        )
    )
    return lists
    # Now append first NaN attribute

In [38]:
list = []
for i, row in df.iterrows():
    encoded_row = _encode_row_partial(row)
    list.append(str(encoded_row))
list

['Name is John, Occupation is Engineer, Age is 30.0, City is New York',
 'Name is Jane, Occupation is Artist, City is San Francisco',
 'Occupation is Doctor, Age is 28.0, City is New York, Name is nan',
 'City is New York, Age is 40.0, Occupation is Artist, Name is Mike']

### _convert_text_to_tabular_data

In [35]:
def _convert_text_to_tabular_data(
    text: tp.List[str], columns: tp.List[str]
) -> pd.DataFrame:
    """Converts the sentences back to tabular data

    Args:
        text: List of the tabular data in text form
        columns: Column names of the data

    Returns:
        Pandas DataFrame with the tabular data from the text appended
    """
    generated = []

    # Convert text to tabular data
    for t in text:
        features = t.split(",")
        td = dict.fromkeys(columns, "placeholder")

        # Transform all features back to tabular data
        for f in features:
            values = f.strip().split(" is ")
            if values[0] in columns and td[values[0]] == "placeholder":
                try:
                    td[values[0]] = values[1]
                except IndexError:
                    # print("An Index Error occurred - if this happends a lot, consider fine-tuning your model further.")
                    pass
        generated.append(td)
    df_gen = pd.DataFrame(generated)
    df_gen.replace("None", None, inplace=True)

    return df_gen


In [40]:
df_from_text = _convert_text_to_tabular_data(list, columns)
df_from_text

,Name,Age,City,Occupation
0,John,30.0,New York,Engineer
1,Jane,placeholder,San Francisco,Artist
2,nan,28.0,New York,Doctor
3,Mike,40.0,New York,Artist


### _partial_df_to_promts

In [61]:
def _get_random_missing(row):
    """Return a random missing column or None if all columns are filled."""
    nans = list(row[pd.isna(row)].index)
    return np.random.choice(nans) if len(nans) > 0 else None

In [51]:
def _partial_df_to_promts(partial_df: pd.DataFrame):
    """Convert DataFrame with missingvalues to a list of starting promts for GReaT
        Args:
        partial_df: Pandas DataFrame to be imputed where missing values are encoded by NaN.

    Returns:
        List of strings with the starting prompt for each sample.
    """
    encoder = lambda x: _encode_row_partial(x, True)
    res_encode = list(partial_df.apply(encoder, axis=1))
    res_first = list(partial_df.apply(_get_random_missing, axis=1))

    # Edge case: all values are missing, will return empty string which is not supported.
    # Use first attribute as starting prompt.
    # default_promt = partial_df.columns[0] + " is "
    res = [
        ((enc + ", ") if len(enc) > 0 else "")
        + (fst + " is" if fst is not None else "")
        for enc, fst in zip(res_encode, res_first)
    ]
    return res

# As an output brings for example (Name is John, City is, Age is 30). As cIty is nan, it doesn't bring anything